---

# HoloMine Breast Cancer Classification: Mammo-FM v6

*Fine-tuning Mammo-FM dengan normalisasi intensitas robust, augmentasi yang menjaga tepi kulit, head bantu Benign-vs-Malignant, resolusi lebih tinggi, dan analisis error.*

---

## Nama Tim

- Isi nama tim sesuai pendaftaran HOLOGY 9.0
- Anggota 1
- Anggota 2

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**EDA dan Preprocessing**](#3)
4. [**Validasi dan Pasangan View**](#4)
5. [**Fine-tuning Mammo-FM**](#5)
6. [**Analisis Error**](#6)
7. [**Kalibrasi, Ensemble, dan Submission**](#7)

---

# Introduction <a name="1"></a>

---

## Overview

*Mammogram digital lapangan penuh dari AISSLab/MDCMI-BC diklasifikasikan menjadi Normal, Benign, atau Malignant. Setiap kasus terdiri dari view CC dan MLO dari payudara yang sama, dan data uji bersifat patient-disjoint.*

*Mammo-FM fine-tune mencapai OOF macro F1 sekitar 0.64 dengan leaderboard 0.72 sampai 0.74. Analisis OOF menunjukkan 63 citra salah di semua run, terbanyak Malignant yang diprediksi Benign (22 citra): massa berbatas tegas dan kalsifikasi kasar yang berlabel Malignant.*

## Aim

*Notebook menyasar batas Benign vs Malignant, tanpa memakai metadata file sebagai fitur: head bantu yang hanya dilatih pada citra abnormal agar kapasitas model difokuskan ke batas tersebut, dan input 2048x1228 untuk detail tepi massa dan kalsifikasi. Preprocessing robust dan rotasi ±3 dari audit sebelumnya tetap dipakai.*

## Metric

***Macro F1** merata-ratakan F1 ketiga kelas dengan bobot sama.*

$$\text{Macro F1} = \frac{1}{3}\sum_{c} \frac{2 \cdot P_c \cdot R_c}{P_c + R_c}$$

*AUC Benign vs Malignant dilaporkan sebagai diagnostik bebas leak, karena kedua kelas disimpan dengan format JPEG yang identik.*

## Dataset

*Data berasal dari Kaggle, kompetisi HoloMine Breast Cancer Classification Task 1: https://www.kaggle.com/competitions/holomine-breasts-cancer-classification-task-1*

*Terdapat 212 citra latih (80 Normal, 52 Benign, 80 Malignant) dan 54 citra uji. Notebook mencari folder `data/` secara otomatis.*

```
data/
├── train_images/
├── test_images/
├── train.csv
├── test.csv
└── sample_submission.csv
```

## Approach: Single Mammo-FM

*v5 menunjukkan DINOv3 tidak menambah informasi: probe DINOv3 ViT-B 0.47 vs probe Mammo-FM 0.52, dan skor fusi turun monoton setiap kali porsi DINOv3 dinaikkan (bobot Mammo-FM 1.0: 0.645, 0.75: 0.638, 0.5: 0.578). Upgrade ke ViT-L tidak mengubah alasannya: backbone natural-image beku kalah dari foundation model mammografi pada tugas BI-RADS, dan fine-tune ViT-L 300M parameter pada 170 citra per fold sangat rawan overfit. Kapasitas GPU dipakai untuk eksperimen terkontrol dan ensemble seed Mammo-FM.*

```
Citra JPEG
    |
    +-- re-encode q75, decode 1/2, flip dinding dada ke kiri, soft mask (background = 0)
    +-- bbox jaringan -> stretch 1520x912 atau 2048x1228 -> robust scale (white point = p99.8 jaringan di luar tepi)
         |
    Mammo-FM EfficientNet-B5 fine-tune (BN beku) -> Linear(3) [+ head bantu Benign vs Malignant]
         |
    rata-rata fold dan seed -> label per citra (fusi pasangan dan bias kelas hanya sebagai pembanding)
```

*Validasi memakai 5 fold dari komponen grup pasangan pasien yang sama dengan v5, sehingga semua eksperimen dapat dibandingkan langsung.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

Notebook ditujukan untuk RTX 5090 (sm_120) di vast.ai dengan PyTorch build CUDA 12.8 atau lebih baru. Checkpoint Mammo-FM (2.1 GB) diunduh otomatis dari Hugging Face. Untuk inference tanpa training ulang, ubah `Settings.RUN_MODE` menjadi `'inference'` setelah checkpoint fold tersimpan. Checkpoint fold `robust_rot3*` dari run sebelumnya dipakai ulang bila disalin ke `output_v6/`. Perkiraan runtime: sekitar 10 menit untuk `bm_aux` dan sekitar 20 menit untuk setiap eksperimen 2048x1228, total sekitar 50 menit bila checkpoint lama tersedia.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q "huggingface_hub>=0.34,<1" efficientnet_pytorch omegaconf scipy scikit-learn pandas matplotlib opencv-python-headless Pillow

Jika PyTorch bawaan instance belum mendukung sm_120, jalankan `%pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu128` lalu restart kernel.

## Import Libraries

Seluruh import notebook dipusatkan pada cell ini.

In [ ]:
import gc
import hashlib
import io
import math
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.optimize import linear_sum_assignment

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from efficientnet_pytorch import EfficientNet
from huggingface_hub import hf_hub_download

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display

## Seed Everything

Seed dasar dipakai untuk augmentasi dan inisialisasi head; setiap eksperimen menambahkan offset seed-nya sendiri.

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(42)
cv2.setNumThreads(1)

## Settings

Semua path, hyperparameter, dan eksperimen dipusatkan di sini. Resep training sama dengan run sebelumnya; effective batch dijaga 16 dengan gradient accumulation, sehingga eksperimen 2048x1228 memakai batch fisik 2. `robust_rot3` dan `robust_rot3_s1` adalah baseline pembanding.

In [ ]:
class Settings:
    SEED = 42
    RUN_MODE = 'train'
    DATA_CANDIDATES = [os.environ.get('HOLOGY_DATA_DIR', 'data'), 'data', '../data', '/workspace/data', '/workspace/hology/data']
    OUTPUT_DIR = Path(os.environ.get('HOLOGY_OUTPUT_DIR', 'output_v6'))
    WEIGHTS_DIR = Path(os.environ.get('HOLOGY_WEIGHTS_DIR', 'weights'))
    CLASSES = ['Normal', 'Benign', 'Malignant']
    N_FOLDS = 5

    BASE_H, BASE_W = 2048, 1536
    FM_H, FM_W = 1520, 912
    PREP_WORKERS = 12
    FM_REPO = 'batmanLab/Mammo-FM'
    FM_FILE = 'Mammo-FM_BatmanlabTrained_CLIP.tar'
    FM_MEAN, FM_STD = 0.3089279, 0.25053555408335154

    EXPERIMENTS = [
        {'name': 'robust_rot3', 'view': 'robust', 'rot': 3.0, 'seed': 0, 'bm_aux': False, 'batch': 4},
        {'name': 'robust_rot3_s1', 'view': 'robust', 'rot': 3.0, 'seed': 1, 'bm_aux': False, 'batch': 4},
        {'name': 'bm_aux', 'view': 'robust', 'rot': 3.0, 'seed': 0, 'bm_aux': True, 'batch': 4},
        {'name': 'hires', 'view': 'robust_hi', 'rot': 3.0, 'seed': 0, 'bm_aux': False, 'batch': 2},
        {'name': 'hires_bm_aux', 'view': 'robust_hi', 'rot': 3.0, 'seed': 0, 'bm_aux': True, 'batch': 2},
    ]
    HI_BASE = (2368, 1776)
    HI_H, HI_W = 2048, 1228
    BM_WEIGHT = 0.5
    EFFECTIVE_BATCH = 16
    JITTER = (0.9, 1.1)
    VFLIP_P = 0.5
    EPOCHS = 12
    BATCH = 4
    LR_ENCODER = 2e-5
    LR_HEAD = 5e-4
    WEIGHT_DECAY = 1e-4
    SMOOTHING = 0.1
    DROP = 0.3
    WARMUP = 0.1
    NUM_WORKERS = 8
    AMP_DTYPE = torch.bfloat16
    BIAS_GRID = np.round(np.linspace(-1.5, 1.5, 31), 2)


CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), 'GPU wajib. Notebook ini tidak dijalankan di laptop CPU.'
cap = torch.cuda.get_device_capability()
assert f'sm_{cap[0]}{cap[1]}' in torch.cuda.get_arch_list(), 'Build PyTorch belum mendukung GPU ini; pasang cu128.'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(torch.cuda.get_device_name(), torch.__version__, torch.version.cuda)

## Load Dataset

Folder data dicari dari beberapa lokasi umum. Manifest fold, tag OCR view/laterality, dan hash file hasil audit CPU ditanam agar notebook berdiri sendiri; hash memastikan grup tidak diterapkan ke dataset yang berbeda.

In [ ]:
FOLD_CSV = 'image_id,label,group_union,fold_group_union\nimage_0001.jpg,Normal,0,2\nimage_0003.jpg,Normal,1,3\nimage_0004.jpg,Benign,2,0\nimage_0005.jpg,Normal,3,2\nimage_0006.jpg,Benign,4,0\nimage_0007.jpg,Normal,5,4\nimage_0008.jpg,Normal,6,4\nimage_0009.jpg,Normal,7,4\nimage_0010.jpg,Malignant,8,1\nimage_0011.jpg,Normal,9,0\nimage_0012.jpg,Benign,10,2\nimage_0013.jpg,Malignant,11,3\nimage_0015.jpg,Malignant,12,2\nimage_0017.jpg,Malignant,13,0\nimage_0018.jpg,Benign,14,4\nimage_0019.jpg,Normal,15,1\nimage_0020.jpg,Benign,16,0\nimage_0022.jpg,Normal,17,0\nimage_0024.jpg,Normal,18,2\nimage_0025.jpg,Malignant,11,3\nimage_0026.jpg,Malignant,20,3\nimage_0027.jpg,Normal,21,3\nimage_0028.jpg,Benign,21,3\nimage_0029.jpg,Malignant,23,4\nimage_0031.jpg,Normal,24,1\nimage_0033.jpg,Normal,25,1\nimage_0034.jpg,Benign,26,1\nimage_0035.jpg,Benign,27,3\nimage_0037.jpg,Normal,28,1\nimage_0038.jpg,Benign,27,3\nimage_0043.jpg,Benign,21,3\nimage_0045.jpg,Malignant,31,2\nimage_0046.jpg,Normal,18,2\nimage_0047.jpg,Normal,33,0\nimage_0048.jpg,Normal,34,3\nimage_0049.jpg,Malignant,35,4\nimage_0050.jpg,Malignant,36,1\nimage_0051.jpg,Normal,37,4\nimage_0054.jpg,Malignant,20,3\nimage_0055.jpg,Normal,39,0\nimage_0057.jpg,Benign,1,3\nimage_0058.jpg,Malignant,41,0\nimage_0059.jpg,Malignant,42,0\nimage_0060.jpg,Malignant,43,2\nimage_0061.jpg,Malignant,44,1\nimage_0063.jpg,Benign,45,2\nimage_0064.jpg,Malignant,46,1\nimage_0065.jpg,Normal,6,4\nimage_0066.jpg,Malignant,48,0\nimage_0067.jpg,Normal,34,3\nimage_0068.jpg,Malignant,50,4\nimage_0069.jpg,Malignant,51,4\nimage_0070.jpg,Benign,52,4\nimage_0071.jpg,Malignant,8,1\nimage_0073.jpg,Malignant,54,4\nimage_0074.jpg,Normal,55,2\nimage_0075.jpg,Malignant,50,4\nimage_0076.jpg,Benign,57,4\nimage_0078.jpg,Normal,58,1\nimage_0079.jpg,Malignant,59,3\nimage_0080.jpg,Malignant,14,4\nimage_0081.jpg,Benign,4,0\nimage_0083.jpg,Benign,62,4\nimage_0084.jpg,Normal,63,1\nimage_0085.jpg,Malignant,64,0\nimage_0087.jpg,Malignant,65,2\nimage_0088.jpg,Normal,66,4\nimage_0089.jpg,Normal,7,4\nimage_0090.jpg,Normal,21,3\nimage_0091.jpg,Malignant,69,4\nimage_0092.jpg,Malignant,70,2\nimage_0093.jpg,Benign,21,3\nimage_0095.jpg,Malignant,35,4\nimage_0097.jpg,Normal,73,3\nimage_0098.jpg,Malignant,74,1\nimage_0100.jpg,Malignant,75,2\nimage_0101.jpg,Benign,76,3\nimage_0102.jpg,Normal,21,3\nimage_0104.jpg,Normal,78,4\nimage_0106.jpg,Malignant,79,0\nimage_0107.jpg,Benign,80,1\nimage_0108.jpg,Benign,80,1\nimage_0109.jpg,Malignant,82,4\nimage_0110.jpg,Normal,21,3\nimage_0111.jpg,Normal,14,4\nimage_0112.jpg,Malignant,85,3\nimage_0113.jpg,Malignant,64,0\nimage_0114.jpg,Malignant,48,0\nimage_0115.jpg,Normal,66,4\nimage_0116.jpg,Malignant,42,0\nimage_0117.jpg,Malignant,90,1\nimage_0118.jpg,Malignant,57,4\nimage_0119.jpg,Normal,92,2\nimage_0120.jpg,Benign,93,0\nimage_0121.jpg,Benign,94,2\nimage_0122.jpg,Benign,14,4\nimage_0123.jpg,Malignant,85,3\nimage_0124.jpg,Benign,7,4\nimage_0126.jpg,Normal,9,0\nimage_0128.jpg,Malignant,51,4\nimage_0129.jpg,Normal,100,2\nimage_0130.jpg,Benign,101,2\nimage_0131.jpg,Normal,102,3\nimage_0132.jpg,Normal,73,3\nimage_0133.jpg,Normal,92,2\nimage_0134.jpg,Normal,21,3\nimage_0135.jpg,Normal,106,0\nimage_0136.jpg,Normal,107,1\nimage_0137.jpg,Malignant,92,2\nimage_0138.jpg,Malignant,109,3\nimage_0139.jpg,Normal,107,1\nimage_0140.jpg,Benign,111,0\nimage_0142.jpg,Normal,63,1\nimage_0143.jpg,Benign,62,4\nimage_0144.jpg,Malignant,54,4\nimage_0145.jpg,Normal,21,3\nimage_0146.jpg,Benign,116,1\nimage_0149.jpg,Benign,93,0\nimage_0151.jpg,Benign,10,2\nimage_0152.jpg,Normal,7,4\nimage_0153.jpg,Normal,39,0\nimage_0154.jpg,Benign,121,1\nimage_0156.jpg,Malignant,43,2\nimage_0157.jpg,Normal,17,0\nimage_0158.jpg,Benign,21,3\nimage_0159.jpg,Benign,21,3\nimage_0160.jpg,Malignant,79,0\nimage_0161.jpg,Normal,127,4\nimage_0162.jpg,Malignant,128,0\nimage_0163.jpg,Malignant,74,1\nimage_0164.jpg,Malignant,128,0\nimage_0165.jpg,Normal,15,1\nimage_0167.jpg,Malignant,65,2\nimage_0169.jpg,Benign,133,4\nimage_0171.jpg,Malignant,134,0\nimage_0173.jpg,Malignant,135,2\nimage_0175.jpg,Normal,106,0\nimage_0176.jpg,Malignant,14,4\nimage_0177.jpg,Normal,28,1\nimage_0178.jpg,Malignant,139,3\nimage_0179.jpg,Normal,140,2\nimage_0180.jpg,Malignant,41,0\nimage_0181.jpg,Malignant,13,0\nimage_0184.jpg,Benign,16,0\nimage_0185.jpg,Malignant,109,3\nimage_0186.jpg,Malignant,14,4\nimage_0187.jpg,Benign,146,1\nimage_0189.jpg,Normal,102,3\nimage_0190.jpg,Benign,101,2\nimage_0191.jpg,Normal,25,1\nimage_0192.jpg,Benign,121,1\nimage_0193.jpg,Malignant,59,3\nimage_0194.jpg,Normal,152,0\nimage_0195.jpg,Benign,94,2\nimage_0196.jpg,Benign,111,0\nimage_0197.jpg,Benign,45,2\nimage_0199.jpg,Normal,14,4\nimage_0200.jpg,Normal,37,4\nimage_0201.jpg,Normal,3,2\nimage_0202.jpg,Normal,159,0\nimage_0203.jpg,Malignant,160,2\nimage_0204.jpg,Normal,21,3\nimage_0205.jpg,Normal,21,3\nimage_0206.jpg,Normal,78,4\nimage_0207.jpg,Malignant,164,3\nimage_0209.jpg,Benign,76,3\nimage_0210.jpg,Malignant,70,2\nimage_0211.jpg,Malignant,164,3\nimage_0213.jpg,Normal,100,2\nimage_0214.jpg,Benign,52,4\nimage_0215.jpg,Normal,21,3\nimage_0216.jpg,Normal,5,4\nimage_0217.jpg,Normal,14,4\nimage_0218.jpg,Normal,7,4\nimage_0219.jpg,Malignant,46,1\nimage_0220.jpg,Malignant,175,1\nimage_0221.jpg,Benign,14,4\nimage_0224.jpg,Normal,55,2\nimage_0225.jpg,Normal,127,4\nimage_0226.jpg,Benign,7,4\nimage_0227.jpg,Malignant,44,1\nimage_0228.jpg,Benign,146,1\nimage_0230.jpg,Normal,0,2\nimage_0231.jpg,Normal,24,1\nimage_0232.jpg,Malignant,184,1\nimage_0233.jpg,Benign,92,2\nimage_0235.jpg,Malignant,184,1\nimage_0236.jpg,Normal,140,2\nimage_0238.jpg,Malignant,36,1\nimage_0239.jpg,Normal,152,0\nimage_0240.jpg,Benign,133,4\nimage_0242.jpg,Normal,159,0\nimage_0243.jpg,Normal,58,1\nimage_0244.jpg,Malignant,139,3\nimage_0245.jpg,Benign,92,2\nimage_0246.jpg,Benign,116,1\nimage_0247.jpg,Malignant,160,2\nimage_0250.jpg,Malignant,135,2\nimage_0251.jpg,Malignant,82,4\nimage_0252.jpg,Malignant,75,2\nimage_0253.jpg,Benign,26,1\nimage_0254.jpg,Malignant,175,1\nimage_0256.jpg,Normal,21,3\nimage_0257.jpg,Malignant,12,2\nimage_0258.jpg,Benign,2,0\nimage_0259.jpg,Normal,205,2\nimage_0261.jpg,Malignant,134,0\nimage_0262.jpg,Normal,33,0\nimage_0263.jpg,Normal,205,2\nimage_0264.jpg,Malignant,23,4\nimage_0265.jpg,Malignant,69,4\nimage_0266.jpg,Malignant,31,2\n'
TAG_CSV = 'image_id,lat2,view2\nimage_0001.jpg,R,MLO\nimage_0003.jpg,L,MLO\nimage_0004.jpg,L,MLO\nimage_0005.jpg,R,CC\nimage_0006.jpg,L,MLO\nimage_0007.jpg,R,MLO\nimage_0008.jpg,R,CC\nimage_0009.jpg,L,CC\nimage_0010.jpg,R,MLO\nimage_0011.jpg,L,CC\nimage_0012.jpg,L,MLO\nimage_0013.jpg,R,CC\nimage_0015.jpg,R,CC\nimage_0017.jpg,L,MLO\nimage_0018.jpg,R,MLO\nimage_0019.jpg,L,CC\nimage_0020.jpg,R,MLO\nimage_0022.jpg,L,MLO\nimage_0024.jpg,R,CC\nimage_0025.jpg,R,MLO\nimage_0026.jpg,R,MLO\nimage_0027.jpg,L,MLO\nimage_0028.jpg,L,CC\nimage_0029.jpg,R,CC\nimage_0031.jpg,R,MLO\nimage_0033.jpg,R,MLO\nimage_0034.jpg,R,CC\nimage_0035.jpg,L,CC\nimage_0037.jpg,R,MLO\nimage_0038.jpg,L,MLO\nimage_0043.jpg,L,MLO\nimage_0045.jpg,L,CC\nimage_0046.jpg,R,MLO\nimage_0047.jpg,L,CC\nimage_0048.jpg,L,MLO\nimage_0049.jpg,R,MLO\nimage_0050.jpg,R,MLO\nimage_0051.jpg,R,MLO\nimage_0054.jpg,R,CC\nimage_0055.jpg,L,CC\nimage_0057.jpg,L,CC\nimage_0058.jpg,L,CC\nimage_0059.jpg,L,CC\nimage_0060.jpg,R,MLO\nimage_0061.jpg,L,MLO\nimage_0063.jpg,R,CC\nimage_0064.jpg,L,MLO\nimage_0065.jpg,R,MLO\nimage_0066.jpg,R,MLO\nimage_0067.jpg,L,CC\nimage_0068.jpg,R,MLO\nimage_0069.jpg,L,CC\nimage_0070.jpg,R,MLO\nimage_0071.jpg,R,CC\nimage_0073.jpg,L,MLO\nimage_0074.jpg,R,CC\nimage_0075.jpg,R,CC\nimage_0076.jpg,R,CC\nimage_0078.jpg,R,CC\nimage_0079.jpg,L,CC\nimage_0080.jpg,R,MLO\nimage_0081.jpg,L,CC\nimage_0083.jpg,L,CC\nimage_0084.jpg,L,CC\nimage_0085.jpg,R,MLO\nimage_0087.jpg,L,MLO\nimage_0088.jpg,L,MLO\nimage_0089.jpg,L,MLO\nimage_0090.jpg,L,CC\nimage_0091.jpg,L,CC\nimage_0092.jpg,L,CC\nimage_0093.jpg,L,MLO\nimage_0095.jpg,R,CC\nimage_0097.jpg,R,MLO\nimage_0098.jpg,L,CC\nimage_0100.jpg,R,MLO\nimage_0101.jpg,L,CC\nimage_0102.jpg,L,MLO\nimage_0104.jpg,R,CC\nimage_0106.jpg,R,CC\nimage_0107.jpg,L,MLO\nimage_0108.jpg,L,CC\nimage_0109.jpg,L,CC\nimage_0110.jpg,L,CC\nimage_0111.jpg,R,MLO\nimage_0112.jpg,L,MLO\nimage_0113.jpg,R,CC\nimage_0114.jpg,R,CC\nimage_0115.jpg,L,CC\nimage_0116.jpg,L,MLO\nimage_0117.jpg,R,CC\nimage_0118.jpg,R,MLO\nimage_0119.jpg,R,CC\nimage_0120.jpg,R,MLO\nimage_0121.jpg,R,MLO\nimage_0122.jpg,R,CC\nimage_0123.jpg,L,CC\nimage_0124.jpg,L,CC\nimage_0126.jpg,L,MLO\nimage_0128.jpg,L,MLO\nimage_0129.jpg,R,MLO\nimage_0130.jpg,L,CC\nimage_0131.jpg,L,CC\nimage_0132.jpg,R,CC\nimage_0133.jpg,R,MLO\nimage_0134.jpg,L,CC\nimage_0135.jpg,L,CC\nimage_0136.jpg,L,CC\nimage_0137.jpg,R,CC\nimage_0138.jpg,L,CC\nimage_0139.jpg,L,MLO\nimage_0140.jpg,L,MLO\nimage_0142.jpg,L,MLO\nimage_0143.jpg,L,MLO\nimage_0144.jpg,L,CC\nimage_0145.jpg,L,MLO\nimage_0146.jpg,L,CC\nimage_0149.jpg,R,CC\nimage_0151.jpg,L,CC\nimage_0152.jpg,L,CC\nimage_0153.jpg,L,MLO\nimage_0154.jpg,L,CC\nimage_0156.jpg,R,CC\nimage_0157.jpg,L,CC\nimage_0158.jpg,L,CC\nimage_0159.jpg,L,MLO\nimage_0160.jpg,R,MLO\nimage_0161.jpg,R,CC\nimage_0162.jpg,R,MLO\nimage_0163.jpg,L,MLO\nimage_0164.jpg,R,CC\nimage_0165.jpg,L,MLO\nimage_0167.jpg,L,CC\nimage_0169.jpg,L,MLO\nimage_0171.jpg,R,CC\nimage_0173.jpg,R,CC\nimage_0175.jpg,L,MLO\nimage_0176.jpg,R,CC\nimage_0177.jpg,R,CC\nimage_0178.jpg,L,CC\nimage_0179.jpg,L,CC\nimage_0180.jpg,L,MLO\nimage_0181.jpg,L,CC\nimage_0184.jpg,R,CC\nimage_0185.jpg,L,MLO\nimage_0186.jpg,R,CC\nimage_0187.jpg,L,CC\nimage_0189.jpg,L,MLO\nimage_0190.jpg,L,MLO\nimage_0191.jpg,R,CC\nimage_0192.jpg,L,MLO\nimage_0193.jpg,L,MLO\nimage_0194.jpg,R,CC\nimage_0195.jpg,R,CC\nimage_0196.jpg,L,CC\nimage_0197.jpg,R,MLO\nimage_0199.jpg,R,CC\nimage_0200.jpg,R,CC\nimage_0201.jpg,R,MLO\nimage_0202.jpg,L,MLO\nimage_0203.jpg,L,MLO\nimage_0204.jpg,L,MLO\nimage_0205.jpg,L,CC\nimage_0206.jpg,R,MLO\nimage_0207.jpg,L,MLO\nimage_0209.jpg,L,MLO\nimage_0210.jpg,L,MLO\nimage_0211.jpg,L,CC\nimage_0213.jpg,R,CC\nimage_0214.jpg,R,CC\nimage_0215.jpg,L,MLO\nimage_0216.jpg,R,CC\nimage_0217.jpg,R,CC\nimage_0218.jpg,L,MLO\nimage_0219.jpg,L,CC\nimage_0220.jpg,L,CC\nimage_0221.jpg,R,MLO\nimage_0224.jpg,R,MLO\nimage_0225.jpg,R,MLO\nimage_0226.jpg,L,MLO\nimage_0227.jpg,L,CC\nimage_0228.jpg,L,MLO\nimage_0230.jpg,R,CC\nimage_0231.jpg,R,CC\nimage_0232.jpg,L,MLO\nimage_0233.jpg,R,MLO\nimage_0235.jpg,L,CC\nimage_0236.jpg,L,MLO\nimage_0238.jpg,R,CC\nimage_0239.jpg,R,MLO\nimage_0240.jpg,L,CC\nimage_0242.jpg,L,CC\nimage_0243.jpg,R,MLO\nimage_0244.jpg,L,MLO\nimage_0245.jpg,R,CC\nimage_0246.jpg,L,MLO\nimage_0247.jpg,L,CC\nimage_0250.jpg,R,MLO\nimage_0251.jpg,L,MLO\nimage_0252.jpg,R,CC\nimage_0253.jpg,R,MLO\nimage_0254.jpg,L,MLO\nimage_0256.jpg,L,CC\nimage_0257.jpg,R,MLO\nimage_0258.jpg,L,CC\nimage_0259.jpg,R,CC\nimage_0261.jpg,R,MLO\nimage_0262.jpg,L,MLO\nimage_0263.jpg,R,MLO\nimage_0264.jpg,R,MLO\nimage_0265.jpg,L,MLO\nimage_0266.jpg,L,MLO\nimage_0002.jpg,R,MLO\nimage_0014.jpg,R,MLO\nimage_0016.jpg,L,MLO\nimage_0021.jpg,L,CC\nimage_0023.jpg,R,MLO\nimage_0030.jpg,L,MLO\nimage_0032.jpg,L,CC\nimage_0036.jpg,R,MLO\nimage_0039.jpg,R,CC\nimage_0040.jpg,R,MLO\nimage_0041.jpg,R,CC\nimage_0042.jpg,R,MLO\nimage_0044.jpg,R,CC\nimage_0052.jpg,L,MLO\nimage_0053.jpg,L,MLO\nimage_0056.jpg,R,MLO\nimage_0062.jpg,R,MLO\nimage_0072.jpg,R,MLO\nimage_0077.jpg,L,CC\nimage_0082.jpg,R,CC\nimage_0086.jpg,R,CC\nimage_0094.jpg,R,MLO\nimage_0096.jpg,R,CC\nimage_0099.jpg,R,CC\nimage_0103.jpg,R,CC\nimage_0105.jpg,L,CC\nimage_0125.jpg,L,CC\nimage_0127.jpg,L,MLO\nimage_0141.jpg,L,MLO\nimage_0147.jpg,L,CC\nimage_0148.jpg,L,CC\nimage_0150.jpg,R,CC\nimage_0155.jpg,R,MLO\nimage_0166.jpg,R,CC\nimage_0168.jpg,R,MLO\nimage_0170.jpg,R,CC\nimage_0172.jpg,R,CC\nimage_0174.jpg,R,CC\nimage_0182.jpg,L,MLO\nimage_0183.jpg,L,CC\nimage_0188.jpg,R,CC\nimage_0198.jpg,R,MLO\nimage_0208.jpg,R,CC\nimage_0212.jpg,L,CC\nimage_0222.jpg,L,MLO\nimage_0223.jpg,R,MLO\nimage_0229.jpg,L,CC\nimage_0234.jpg,R,CC\nimage_0237.jpg,R,MLO\nimage_0241.jpg,R,CC\nimage_0248.jpg,R,MLO\nimage_0249.jpg,L,MLO\nimage_0255.jpg,L,MLO\nimage_0260.jpg,R,MLO\n'
EXPECTED_HASHES = {'image_0001.jpg': '18b80e16f8a5b2ae819d5b0ebae6714124c8a1888e667a9c9f7d1b232cbee694', 'image_0003.jpg': '61a5a28584131bc34d75572944bbd9e439eb4587442d691db2faf8c8c3a285a9', 'image_0004.jpg': 'c9e9f767447dbd9e9c8a26317dae00d50216d091efb835feb0328a8c665db1e9', 'image_0005.jpg': '93443a93de9836487442478cc4f467ab84898a7d066d439b81b0f3d891885e59', 'image_0006.jpg': '08c262f0dab5dca8ca29d5c2a96d053ae2dff8d1069e03410a34c3d006990571', 'image_0007.jpg': 'd0f6c2946cdfd35008ada1a919cc985cf82e0a8c0ecde1bd258bb4d7b8e26c4f', 'image_0008.jpg': 'da52e5962b40acf99341790176927274ef6e30042f319bdbec3c5101d02a138f', 'image_0009.jpg': '3ea780804463bb448a9ceed21cb2515fe35ab8d03fa5083d5dd2b4caff1ec06c', 'image_0010.jpg': '6261b797d86ba34552cd564691db9dc56b85e479317a016764224cd2b122cc7a', 'image_0011.jpg': '88d79543aaae06bff3632e324d181a80be8a276a50b87c3ddb6541368a20a0c6', 'image_0012.jpg': 'f9ce86387374bea7ea3741bf08aaad8b7b555fee242f3a1b63f888defff98e4c', 'image_0013.jpg': '2f4bdd7eaa0d7b4ef464a1e82c3f030c3aeae293a62fea16647de06121d66722', 'image_0015.jpg': '253aebc1b57447a55f43f442df6b1985b2fe7289ea64e874cde101277656c0dc', 'image_0017.jpg': 'ec3f3777fbd60551130fbb5b9fddd9aae1db7a59cba6a16c1130cb90c22afd27', 'image_0018.jpg': '17fdf8f7113dc7d854ad2b2c6d967791d975e55f8089d7e32f207f469b78cb6f', 'image_0019.jpg': 'f913f2ba5110eadddfe5aa6778fcb732740a38b13bcc365e0cd2559dd866b84b', 'image_0020.jpg': '3508ace0efde405718c37108ffefb1bd0ab5970f85535f5f9e76497f8ae9fd66', 'image_0022.jpg': '23633dd1c67ff658354c21bd877e67a5d4bfd1fc83575c44383976d1f21cb91c', 'image_0024.jpg': '1dee372a7c3c793a50c72c345bafe6cbe7166ff8934b132e40d703b431715963', 'image_0025.jpg': 'cb22e1b3fd7942ba87653bb023e5f5b8e77fd6ddca830fb7a21abad83329a4b4', 'image_0026.jpg': 'b7fa442f986f4ffd1a9ab0ac4f69a36d0005c1fd5438c9ab72b997082322af75', 'image_0027.jpg': 'c1317a666f586d648457d6e10b9610a547642bf549df2b04060b643ba1c05a16', 'image_0028.jpg': '91de3c500a6bb1eb7a07b19ee40811ffca4b5fd84afc088766f4a6dc2976a17c', 'image_0029.jpg': 'cde6b94d363df88749c06ca23027af29923f40275b0cf2b12a84f9ca6bc7985e', 'image_0031.jpg': '323b653fb6a6f11f63bcdfd239e31cf6ace9cdc52b9dca665f867f36be21e6b9', 'image_0033.jpg': '585978279ec3dd4c33940d8eb446fe7962feaaa4b4956ca7dd2e944ee1abff12', 'image_0034.jpg': 'cbff8684ccb3319ee09686267286159745dedf8f21ae2ef93b3affaf783c9e08', 'image_0035.jpg': 'f753e2447ca8bd782d296391af32d930caa62829d11a4e166f84ee80a1d7904b', 'image_0037.jpg': 'd0268d07f759fd9c59733ef1abbd9362c379c7338205fbb1688d928ac96fc573', 'image_0038.jpg': '584495a315669ee2e67a296ae2fb273faf00936c2ac57a4c8479b229e8f5d20a', 'image_0043.jpg': '812f6e8134bed66e1f4f8ff5444aeba9505e1506ce5d69682390767df3167c01', 'image_0045.jpg': '854517fa022bed4d6d5fa31465fe324f08961d62dfc052961bf0a84fdef1d732', 'image_0046.jpg': '3c4253e61cc3a33514297397b7c39031f5a2995392b43534966c5499f9e5d103', 'image_0047.jpg': 'b2ce8d032ecb544a712b579dc71f74f8b7d1d6f6e8d17d87aeec7464abd0670d', 'image_0048.jpg': 'b7a6716f970a95b6a2300f53b97a333c7d1db5a273d97288e9c9d678804b0807', 'image_0049.jpg': '8eb4fbf3f50670c0de451b01d2a700b506bd3fc7ac65b9964e1be6d6948f5914', 'image_0050.jpg': 'e39b64171e785cac91a9c72d627e87586c1e20e5a2534a25505887eee1204bee', 'image_0051.jpg': '1016f200080ba4265675812df990c19be88fbb8004202fd6564b5fec38fb96d7', 'image_0054.jpg': 'ecff367d3ad1f5138e473573e15daeb463d5cc7a4a1ee374373b6271dc954d57', 'image_0055.jpg': '0e26b84910d2dbf6e3c77e0ecd80fb224654b008cf63dccaf8d572942f97b1ff', 'image_0057.jpg': '04088b024be6dd5e00705aef60cf8208bac11b0a7de7c3fd6b698f448084949f', 'image_0058.jpg': '444fb4fc42b4ab662a8732c3ad66e0cf0c41383f2a205e1349ffa07326b43a45', 'image_0059.jpg': '331ed319b37e420587a440df332570804d5e52502882bd6d29b61830ef1bcdac', 'image_0060.jpg': 'f7475b22a065425c8ff3f6fdb3e69e4e4f89ff4ad7ca666fa49dc7765f84a830', 'image_0061.jpg': '762e0803f4fe0ba2ed00fbe184237e6ca484385e4a7936c7a3ff10534f784e3a', 'image_0063.jpg': 'c45484e7d3271436d1a307b1213dadfb643a880e7c551f615db78659f1ad9266', 'image_0064.jpg': 'c207e40ffe99e05112f7b742e55cb5cc9f8f9b77f90409df3a695aeadb5c8752', 'image_0065.jpg': '72cb9fcd1269551fd5261b3ec0472d121da15c1b456b131206785081ae46d8b4', 'image_0066.jpg': '87a8e63f0c0aa3b4ada2d1d2d54edd4a2f79a31f26bd8331ad4780baa7dc1efa', 'image_0067.jpg': '5c5e6ce0553862eadeffb450d1090625e1e782eec52f95860d4f42e0b8b6e1ea', 'image_0068.jpg': 'a5af717c05a27fd59424b78f6b31e78651fe74f77d22c5cf15880b69bfb10799', 'image_0069.jpg': '993f06c2feec4fd089b6bf549fd487c52341f76af89582d76d0b915c6dbc7bce', 'image_0070.jpg': '3bce39c898e21eb000c7291ad6cd4f6bf332de35170ac1d8ebb73e0cab4b3bc6', 'image_0071.jpg': '198288a9198123c70341ac5f4725e570eb2ac48e8a0f0d8e27a2a08f89998baa', 'image_0073.jpg': '8fcc2287e5f1f710a716dac431532163a9c8d4ddd81a540eb6946e3a36109fd2', 'image_0074.jpg': 'e4975f81ef8fa2ba89481c8514a07b234619886400cc85812d72b7b0950a89f3', 'image_0075.jpg': 'e2868a867acd5a15e049127d7249ec6c0428e89d83b4508d033f488abb713f4f', 'image_0076.jpg': '63086660e7e3d20c27e49334094f9899ed2bcc49ab50cca0c5dcb1e29b79d29b', 'image_0078.jpg': '5182a0b3825c90a844cc3aeea484ff9ac339f1e400a2b6633a0e17511e687b72', 'image_0079.jpg': '17ce38a4e56e87c9848706d9f1d7b11418a9432711138ef2f3543585fc9f9357', 'image_0080.jpg': '84e20e2689badabfd62dc34c88af9e80e2350e460a9de1c904430c87bb9e8d0f', 'image_0081.jpg': 'b7c76d20b27411cecde82c646701ce23f2af5072fd93627c6f60a155c7cb3a9e', 'image_0083.jpg': '4738ce80d0d1fac9434eb8fb9122ca4b452dea49ac156d2c6b0c8167997fe288', 'image_0084.jpg': '1cb36fe880c02800186f10100fbf00db39fdac2d8edacdda810ad85667ee9749', 'image_0085.jpg': '9566589b6128fe1083ba4d4b2013e6d0be713ba0b12eb42c630e4bfc88db1337', 'image_0087.jpg': '460791dadc63c8546055cdf600d311c5bf34c9a5d567323ca77e2399019e488f', 'image_0088.jpg': '00b618205f81f66ad49d2a3a56a85582229604951bdd90fd1c232d0a17b668d5', 'image_0089.jpg': 'f9925819b829ad1a800f484cf1343ec194fc38e3d392686751a3dadcbdd6bb55', 'image_0090.jpg': '553f67b7cc289391d5c97d57b403442127b2245bed37f58e89e37ed2e32f2599', 'image_0091.jpg': '7fb60a52911fa8a79158ff6fb53d4f8f2c4f6b202f7dd2c2e388d646bbbd3be9', 'image_0092.jpg': '263feaee9493e161d97babbe73de4ed6764c32f41fe0524a948dd2ed2e34ac13', 'image_0093.jpg': 'e0bf33e8351e469f49e2b6466e51b4e46a63cc9e5a1c56e81a68e70df2510f27', 'image_0095.jpg': '8bc90823a4eed6ad233ec866891b83a9715393ba2ab2cdc76c569e0e67e41b86', 'image_0097.jpg': 'c6d84806e764ca1ff59bb9f4cac6711d52d05535ddc8e42de8ec30100df9c401', 'image_0098.jpg': 'a45edad3171d732178ec3728a2203a42c795b9c272ad305f9c3a5de675369458', 'image_0100.jpg': '9af63506a5d339748ba393155ca1003f38ea96ba2fde9cd99d780ef37309c024', 'image_0101.jpg': '750467e55dbfba07dc49064be7bc5aacfaaa5611fa2886049e8462f38c0f87b4', 'image_0102.jpg': '749fde3fcac7ce8bcc150bee9eb6548164223254b02f4ae283c8f44e103dd89e', 'image_0104.jpg': '7204d0db2ebe4bf6199466d2cf5f910f9511fc608501bcebc3266742ccfce5ba', 'image_0106.jpg': '4e94be23a3d9daea5ac5023c153d560657e5215231dad8ad45d706ae705ca420', 'image_0107.jpg': '1fd228c0756b17b4cb39ebafa0c7c974901984a85cdf410650fbf7646bf5a8e7', 'image_0108.jpg': '0673d0e412a9f65eb2a1ab2ab022969584b1ffc518fd5446168da1c574bd0715', 'image_0109.jpg': '7198415cec62b8f819e6a61c402067ad22ec20d5b8705e9b1626ab0d0c85feb1', 'image_0110.jpg': 'f9727721e175b2fc7ecc1d59b536682fde8ee96d0c3bd956932d6e64689198bb', 'image_0111.jpg': '755df504f7412a6e7cd8e6715f93c4282dc1d1b705ec223d1ec8e0a0ce7e2bf2', 'image_0112.jpg': '39fe5a50e3e3fbc99a57e3d9994ee8999b0951c890afa6f4694e60b8fbccb5a2', 'image_0113.jpg': '4d70729a894e37ed165a817ef97b6c43ac205a2276bfb726cf02f8c316bf6e1b', 'image_0114.jpg': 'f7e9dbb38fb32b3e770b37126a453e24e691d1425c306a5bd6256d485c791af6', 'image_0115.jpg': 'c134985474a1fd219b767ea07841e01407b5c873370871a83c51b156f5d86d7d', 'image_0116.jpg': '02b47a810bd10ca56f7cddbf6f31bebc9893207c86aad9bb08c8d26e880a2a05', 'image_0117.jpg': '81e16b06a364558a2553d5424685e0b009919aaf18e9d1a6008114b881d2a05a', 'image_0118.jpg': '100078db1783a2c7658facd15739992bbcb9fac3aa579ed31bc72ca86bf90925', 'image_0119.jpg': '7fe5ceedf5270ec3e861e47d268668157eba0349bd36d9e0abdf7c4b9dab7db9', 'image_0120.jpg': '448a55f52839350cf4b59bc2c8335d6b8a66472b199df227b58fc89f48be8993', 'image_0121.jpg': 'dcad7216f4e815a8048804d7417c4e228a65c2c40794deb142d4f3af8abb5ef2', 'image_0122.jpg': 'dde6eba8cba18673e0819b9d46168edcd8b2b45f07e1f0fc723221983a4b04ff', 'image_0123.jpg': '2be43c7ebfd077b26f0b813946a5b68b7af2eb3625f6ee382d19bc0a79ee30bb', 'image_0124.jpg': '270697db484de7eae2f4b3f00bf6ec1ce092276d564978c9e7cc9f04200208f0', 'image_0126.jpg': 'bde38bbe67942a93141307b631f505b9f442e9c3298011c4088e1c6ff3456dca', 'image_0128.jpg': '207762d17137dfdc72d5670df172adf5573fda1ec992fafcc88d8e6674a301b3', 'image_0129.jpg': '5e5c64a080b4d3f84ea80fcda1e5044e42ee1b5eeedee90c466321954c6d73c7', 'image_0130.jpg': 'baa9a91f204856f10c2f00e55b55515d1610dc88bcdb1d7250c974be4fb9ac50', 'image_0131.jpg': 'f0738c28a36751ea464c0357246c4e28d1e46e521263fcf07d62025d18589115', 'image_0132.jpg': '9c1943e92403417738fe1448efbf522bdbf3d1d194544eb70346d8c82474b3be', 'image_0133.jpg': '1251eaf6bff2b10fa535364d825fe53b4e0d7325ecc7ff6330a53cdd3bf93115', 'image_0134.jpg': '1575416ebc6a2c4c2be5e610a4200436b7beeab98112c98ed73bebabe3bb926d', 'image_0135.jpg': '5da76fbd9b3fe62fe97026955fc191d5ea98d8d1c156030e9c52565bc747a350', 'image_0136.jpg': '5cee13e30c0e84717ef15706a95d0e3167992ea4e7b6c3bda6407dea9d3a1bd3', 'image_0137.jpg': 'b73cf941bc731b56da7b47c8ab5a317fbfaa94df290e7ea9ec99fb4247f12822', 'image_0138.jpg': '5f5632df39ba1b8db0e0782c401c0a9c4765595212b43691a79189400bff9e3f', 'image_0139.jpg': '6763b517b07bf3eb24dbfa9320e1626680060416364b69912c11da313ad0b4fd', 'image_0140.jpg': 'db458b6add2bc2464f0965d270234da0684df499dae8b4515be260b3334adbaa', 'image_0142.jpg': '9accc4341596df92968c81173ea135bdbd33bad713f144d3bb7c332096df9172', 'image_0143.jpg': '11b68032b00dcf4aaa84433311057d70d234013b7a0ebd1a0ba22352459eb3ce', 'image_0144.jpg': 'e1f3879fec676ffea4f44b1d6011d4976577ab0a1b524136b4f9399bf9bcba5d', 'image_0145.jpg': '67b9245a61ef01b5cb2f7379424cae973258c2a04c498862123fff0dad24a405', 'image_0146.jpg': '1815a0538833401a48a9e1af8a177a1600c6541ad18de848075edf63acae513f', 'image_0149.jpg': 'f878009fa1021dd9b8090b2dff64d73eaa38ace157b949c693d51b5b6cf58943', 'image_0151.jpg': '99528d2fb3750c7d163717f13f1fe649c01339acb9427a400cdffe03f34f5946', 'image_0152.jpg': 'e9244179254a424130f96e1fd7cc89be663d98dd91159e6cad718f78fe5fbbc0', 'image_0153.jpg': 'eaccb37cf281c335dc59b52cf2a733c2e26ebd059dac211ad74e8cb620873ae0', 'image_0154.jpg': '6a3065985497b0cfe7ae58859a5669954856eccf4b98c9e00f3992029e065697', 'image_0156.jpg': 'd8493d41078b81b0543ee7bf36282f8ec8c9b639f2bf2fe61ed6690e85573e9f', 'image_0157.jpg': '531fa4d4f546030b6d10382ae00576310986a0f4c57647a7ce996ad04738512a', 'image_0158.jpg': '3252eb370394483c1dfa4503423d94c2a89edbf9ef51102d32eca8ec09563745', 'image_0159.jpg': 'f8a423c312b048c73490fffb6dc880c1b3bba757aff7f86d9f6b726413ca0ab7', 'image_0160.jpg': '9de15827ab160d54e9b181dfab86e64b16c4294a747cc62f87084859911477d9', 'image_0161.jpg': '996a5097d229c46e97031775bc15402eb3291b0dc0a843373314566121cabf45', 'image_0162.jpg': '68773702f8421cc3940e0d460fe7686d96a9771e01b10dec13f85298bcb48401', 'image_0163.jpg': '1ef31666305320d91254c37ab6d6559eac6da2c5172ecabb7ead9cb74dc539fd', 'image_0164.jpg': '058546b83053cd0c0cbdd33e92e2d4ebe5182344769bab3b3e64d4d9346f88c1', 'image_0165.jpg': '0cbb5147d223a19c447de2ba6076d3e72e4ab232cd80666cc2a761943966f0ab', 'image_0167.jpg': '0b76f23c78cdff4ef864fb3b521d7eed60ca63bce96368aef1dcb170119e53be', 'image_0169.jpg': 'e84ba2ceda456abac075a0facc07bbd729d4421eaf7c54b1ca1e5fef4c010aed', 'image_0171.jpg': '54e6c7c72cc00ed00aa284a06063b4fd18d9677bca57d03c5fefb11dc0dee5ec', 'image_0173.jpg': '93e3a1bb8bed9ee81566b693cf7bc9062282e5c44ca7ed98bfbfaef77cea954e', 'image_0175.jpg': 'a5a05f91d714470746dfb87aa35f5f05a1ddd208966a39a2ea90896a029cf3de', 'image_0176.jpg': '154d25e10fb1f9cd14aba3732a4ca3b7e0ec7eaa29d43566497c1c4286ffd9f5', 'image_0177.jpg': 'e582cd91bf04ff31bc22b372e7b17e3f810a9a65fb34b1dafeb9e947391213fe', 'image_0178.jpg': 'ddb69f5d4defedeca7a2776c9266a94c91d189cedaeda106395ea6a9f2e0c432', 'image_0179.jpg': '285ab79a10186d77161e7a8b6c38f3b0939fb4802d008946a9276379d2087650', 'image_0180.jpg': '71daee1ef23a0105ba8e6236dbb64cd9680cbbf7bf1a5fb8f665894acf30c7d6', 'image_0181.jpg': '2bc505a9e10776a10afc4f14801e6e69b682a58634a4e180dc82680660a6d060', 'image_0184.jpg': 'a39c10e70f980668c1e670600b3621768aa046a14a9c5467e8da85ce2fa8431f', 'image_0185.jpg': '13799ea735afc10fddd715011a8b74aefd2cc907e94807ef6bbec14a2f881299', 'image_0186.jpg': '8b911452bd17d94d395ae4ffa40dc13c80a51286913715efe7b0e52ee5236ba5', 'image_0187.jpg': 'aa3eb14a83376962691c60689d3190f4d1e791550e76f926b4e1ab2838b1525d', 'image_0189.jpg': 'bb4bfa41e43705e41551b9830234cf98f3a58a1649aefcd3d7074a76628ab52f', 'image_0190.jpg': '2e8d451810098eb1c98b7f8ced98b006a4e14dc235c755e106fd2c129408f454', 'image_0191.jpg': 'd5cdf6143e016cb1a01ef9785fdfbc65fbff23084d9863e28bf61e15edea475d', 'image_0192.jpg': '8e4ec96ce89f2c416cdaa6ec68f4f52d3cb0e6be03be8acf714d5eb79f4fcb24', 'image_0193.jpg': '143c297ddb94546ee349d56442cb498239c2554edd6f05b7472bdcf2a4460f21', 'image_0194.jpg': 'eead7df290bc415fde6f5efc74e570d6889ed65f4d458509c643ba0494bca1a1', 'image_0195.jpg': '937fd6969c7f26d9fe548c7d6db6d00b5f2894e5e767ca393efb6b4810261849', 'image_0196.jpg': '97fef48284232d690663919f1d78035a694a19b168229d59c661e71ce3415984', 'image_0197.jpg': 'fa4c53d888d1aad19bc366f3e06317c3efbd35a82528d62df33f080dc8240745', 'image_0199.jpg': '79b64c7b210e7672c5788d8713276b6545d77ae650b8c7f043ae696f2733483d', 'image_0200.jpg': '6420fbd1cd958d56289caaa5c3d5fb9f852172c62018cba44fc63aa7b03b3380', 'image_0201.jpg': '82bfa373d5a21f7e9a0b0945a54e9e15521ba67550d501ef4f6786508a435372', 'image_0202.jpg': '2873ef3a619ef575f00b6c7b2aa493cc6f153daa36c0fb4a936ade02195f9127', 'image_0203.jpg': '3910b2eb48b261d86e4fc10f1a9488814d0f0f6ae480dda6d909f607b5b082b8', 'image_0204.jpg': 'accb1e4d42ff380eab6923856272efa8b765e19c3f91fa72d774928585adb7a4', 'image_0205.jpg': '4fe2b2238587e814771aaac1ad881bd983fac620c8071ee70ef54f5a034b1ff8', 'image_0206.jpg': '28a1b5d455acf69ccc9cedaa8398f01ac2ce45c5ad70d2b9171a3e89d4bb6dd9', 'image_0207.jpg': '674c54074fe5e8f5afe71642740560dcd3ca7a6c49f49749af565ef173981b0f', 'image_0209.jpg': '63476c91174df8b85c703441f8a11bf3e58b2af828fd8f5c3fa7f21c95f7ddbe', 'image_0210.jpg': 'da03005e6b3f091ffd8fa0da4ae9f012b45da28a0c491f2198cbf07a9f5d7663', 'image_0211.jpg': '511f1a894e8aa67e8cd9933764d6c20f48178572c7fdac35e4c28a7b731eb615', 'image_0213.jpg': 'a9b98ae311e2956066981e4ed653299dc146685056b28b96a6e6d2b140996095', 'image_0214.jpg': '174ecf5775b7a7367662699235207cef5add092f27ac2a75d5fd74f8f491a59a', 'image_0215.jpg': '3b4db766eb338e125a3ee7fae7be270cebae5b4080ca00e438870aff25899147', 'image_0216.jpg': 'dafd333b4ea06b5d4fdfc88f5ca550e63eb499f4c4f462372f37ba0662e88d65', 'image_0217.jpg': '5681ca5ecba4f082b5f5da2d4a9e33eccb855091dc1900ce149fb5c4a6660f24', 'image_0218.jpg': 'b8611020877063fb14bfedcc12c080e86ae705e48a7a0d02ba2f9b458c6b55d8', 'image_0219.jpg': 'f386646e7402bf9dcfd67783388094e03a7dfc78b5a90789341b751d55801618', 'image_0220.jpg': '8821c27abcef8606db84e232ed9f413244118a03b6180c818a6fe84fd9ed98f8', 'image_0221.jpg': '9b4560e41ec92d4ffa7d5e6f39c90782988d68f91d0997741b93b39dd31c95c7', 'image_0224.jpg': '4641c871121db6dce65b9f40ca2cc6feb9b960f070471e49602e9edc0ac8b2df', 'image_0225.jpg': '71a8838aefda65612ce6ab7b1be71be0f90beafe70e2e4a8263b6c6a70242f6c', 'image_0226.jpg': 'ba6d688873a684fcf14def122cbaa349e693a3383be159f7feba17fbd0633fd7', 'image_0227.jpg': 'f0d32fac3e94b6c65c3091bf42847f2d197ad7ed5463a96eb13a0513ccbd2041', 'image_0228.jpg': 'de4fd4165fc1f0893b12aff1ad082105357c8022c4ac422627cd2df3f6206b77', 'image_0230.jpg': 'a1d28379781175bf93c1a37664c7f5b8024c026cd645f982a95c38b7a02e6427', 'image_0231.jpg': '275b357e71208de7cae0a4fc7a74978ea54cd270363f2849abb3a4dc404aa0ec', 'image_0232.jpg': '130938370978e0d978469651df01c1f28706cb7de69107f34d970284452e2652', 'image_0233.jpg': '37a66e3141de9a7f855c411ec541a265b63b428fd81df90b1bb0ff493c961655', 'image_0235.jpg': 'ceba7214d7a48b17e6380b662d798286b54001563a15060faf62a685248e45df', 'image_0236.jpg': '890d76bfb028532f396e43ae406e67fa53b39a4025d0683407652ed23b7c021c', 'image_0238.jpg': 'cc4c539b629d3da6dcfb5a4a1ce46c53bdd88d941c651200bb29d8dfe4a1d55f', 'image_0239.jpg': 'cdcb4372b6ad5e65d0f0d9e90146a930f56f3c6cfe8cbe410b8e1cd8a96b1f8d', 'image_0240.jpg': '04b39d75d0b42f101e680ea2eeae67231d799ae80652687b1e6710760135d222', 'image_0242.jpg': '5c02de7bdd1ce96d0ee4747616e51778d3163decd0f0dfe60d34c1d5f2d50465', 'image_0243.jpg': '9a7ee7411e8e4e74bc7841883e24425667a9b8ccc00832974946869a3422d2b1', 'image_0244.jpg': '3847f1dd969924fd8ff3a8e9f2a26f1a286b71626beb4b73371abc16dd447bb4', 'image_0245.jpg': 'a7469365761ec6a0e45755d2ec2bec1b4d4d4ba10a5810b42c38e1053b47ad80', 'image_0246.jpg': 'a74bfa60de54f9c25e3ffce6c94eb69c4a6ed9e70ed6b37e0841a5f730265b1c', 'image_0247.jpg': '2fb405e950f8aba5627f106d8403f39374b7ba0e6b0367c99e6b1e87a1b47564', 'image_0250.jpg': 'db9b0b023c41bae2794850efccbfccdb4a37dbd81ccd91b4349949c4658b2a48', 'image_0251.jpg': 'c2d4da10684353bbebf3e0e692ec995f92edc3abd074eef3ee2f6043061eac2f', 'image_0252.jpg': 'ebf244f066851cf3f0b97e2e6e7721e04ad72815baed4315a71bf6b28da4a1f5', 'image_0253.jpg': '8c119e0dae82a4a024818130f93090d7521fecf2ba89518fd4f9e6d4a6d60169', 'image_0254.jpg': 'be218affdde07c1735f61591b2260e446746470484e4eb2bfcdf162a1d3fc044', 'image_0256.jpg': '07d73fd6ad0a4f059fbf44a1792ac081c3e3e3a2c1bd880384199bbc4d4f692a', 'image_0257.jpg': 'dbefaf6a74e52bc488a540d4a2cbd56993a93f3f4f18a4b1880420430519ac20', 'image_0258.jpg': 'b2e546bcca0df1475300f0fbf174d24ccf5b36ca0fe6da2baee6849f50a0f6e1', 'image_0259.jpg': '1d4e03ea118a59e0a189f53382f7b335ea645961ea3cc23cd11e63c02b573b79', 'image_0261.jpg': 'e38458d57fa2a4a5e9a50935773f8e4b00add566607729988708b002f6f34a07', 'image_0262.jpg': 'ef20c445d3a8030d0ea2b7cd00eb84af3c384bc84a5433bc072de793a277913a', 'image_0263.jpg': '0e584ab6475fcf1012ba569900c5293074054022e80b687760cf29139f65790b', 'image_0264.jpg': '8686808716c7deec816032e8e72be91fae048e6a0a95f4a274808c0c7837c599', 'image_0265.jpg': '14889d3c9728706d63b5d298103b1f39baa46d294c9d502da0467f0e893fb294', 'image_0266.jpg': 'e254bc30af42d3d4b784e0260a57de7f59dc008a73198ceb766f8267598e00fd', 'image_0002.jpg': '54872b634596e2cbc5e00589a83ff28b0991da78c414db51cfdef23a76d4a931', 'image_0014.jpg': '29efbf5f9a6a5bc18e7eed25c1e9fcb6d6372a4f992a05e0aef1f49a6e50ade1', 'image_0016.jpg': '966fda7d850ff0c5acc110aca051111e4d734684a704cb207b6586d9fc445ed8', 'image_0021.jpg': 'fed146dd6d399289422afa19c16ef564b9d2671ba738d09d149b230541e4ea73', 'image_0023.jpg': '92769be5947163f900f702ebb55c12242cf87fb6c61fca3e0a6ee8faea4f88b8', 'image_0030.jpg': '7631a3516a019d2a11d176dd11fc64a9ca0ad82598a08d1d59ff096e430db4b3', 'image_0032.jpg': 'b20b52e271b4bae7f019d126e2c94f9c903d73a6d944b6d4a4b617b043c25901', 'image_0036.jpg': 'b1baab04b28c2d1d935d8e9d57bf0d0b19b9d7f40dba9ca072250f8bf394dbd9', 'image_0039.jpg': 'b1895be4f4df44f41f52a63531cbbf94415c5d1e46ef3517bcd018e7db95790a', 'image_0040.jpg': '6fb8ca859ece500ec7edfef6282da4899742926ee9d0fa787696779bdaf20639', 'image_0041.jpg': '0d9936bba826b23a687edb6dba3f576e63f5665e27aebdd9f781f13211ab651d', 'image_0042.jpg': '818de989248a82df634b333105b3965e0968149a13534679681d313ded846bf5', 'image_0044.jpg': '3c5df1ef7baa1996fd3479fedacf1fa0ddb959766473c1391d6169738c89c8be', 'image_0052.jpg': '236910f91d04b41d2748d278c758c4e2f200a5753ff2341a17eb4f108e024b43', 'image_0053.jpg': 'ae08f787ddf2c9638f391b59b1a5ee9128c8d7319e41f8e472655bb1c76f2f96', 'image_0056.jpg': '386769263c42f8af0dd3a44f462a0a6fa9317e23972c8dd5a79da3aceab003c8', 'image_0062.jpg': 'b91f945cddc81e654df8df1a8ce08570681454285b2bde7240ad0f7f5227e303', 'image_0072.jpg': '945bbf74be95f1f9ecc5f8fc122188cde2535f588b162abf94f17f2264f29087', 'image_0077.jpg': 'a4fd1018b845ab57b8a84fb5345566aeab7e595fe1b1a99e69356ec5bcd4605b', 'image_0082.jpg': '506f27ce8a0455340bdff78908fc16d73c248d5a5ffe61b288739eef0a070160', 'image_0086.jpg': 'c8bb74fb942d374ef6476da2963b8b5a4c00922af2a5c10c6add934685ef23f9', 'image_0094.jpg': '950aca7d5d063ca934f5f1d57b277fce6f8efdc07487963ad59cf2bd2d20d996', 'image_0096.jpg': '56baa4b40bfd0d947c89f4149af4997bc9d17c3958094f68bfc99396717316d8', 'image_0099.jpg': '7cc75e91615890b9ad8a08a116b73a162d78e667682d4cc92d794e60afe60169', 'image_0103.jpg': '298c7a0d4921f13231ba9505669bf1b0803b4f656fbc778d54e0d6df18983852', 'image_0105.jpg': '954d08e4679a430398d2478978df7a774b467865464d430301bc94d333e34ef7', 'image_0125.jpg': 'b4bdf5702fc4c23840eaa5c8b9b15bbb077fd76c7a622692f216e1ae1a144b17', 'image_0127.jpg': '20242ce85d7e15126d759453440ecf4732beb69c6d924fdb9798c15ff604520c', 'image_0141.jpg': 'ef4d5365f36c21bb5f151b45ef9deff8e5ac31cb63ff4f2bbc99b0b73958a523', 'image_0147.jpg': '5c62a191934630d83d258a144fc60e0a59348ffdaea58e246644d6adc5465618', 'image_0148.jpg': '8c1b7659d78daa189484feb51feeb9c07a5194ffd36a71456b0fb860c6f2eaa1', 'image_0150.jpg': 'd6d51be1bb1bd3f0b8b7307d7968bc1956231012070013c6a0976b9ecb149756', 'image_0155.jpg': '5cee1cf15c68c394c8db0cc3109a10f556f2891810ec141180bfc8a370e50f12', 'image_0166.jpg': 'f52fac87940a3bd69dad7b5d206e8d8f726fbf3d782cb64a7aa3da9718ebc790', 'image_0168.jpg': 'b896de1f45b3ab6631333c7c8b07334d6e9de58418652adcf380dc53861f92da', 'image_0170.jpg': 'ea914735ed0e7a64e17c11382317dd4dadd18a43561e2579fab509708585783b', 'image_0172.jpg': 'd0c813b1087c3b405636123b10ee8de3d4031c970935c3f9933c8f34d1d6cbfc', 'image_0174.jpg': '6aa0da42f3a69636d8df8cd6d462fef319c0227779ee8623086bdbdf9370c888', 'image_0182.jpg': 'fef6fcfd7eb25d09f1c3c9a0a158c32d90818e214f4a906632897c4c2a837dda', 'image_0183.jpg': '0dc72cb231b21d09b84f7d8573c237c1369bb7d23cbaa614ec9bd57315e99bd9', 'image_0188.jpg': 'bca1898f4c34e649b49293d5e44c45ba6d6e9c38fea87cea71064cf53a496854', 'image_0198.jpg': 'ec1d7f0fe6cfdc6ee96347910d8d1ef54e8ec178f02ad36896df93ee4374b559', 'image_0208.jpg': '09445c036773b4a0e91777e72f74df56e11d1bddba991db4be39696d4c120bb9', 'image_0212.jpg': '9814901ecd0c5d2737ff6bccf15163329ae5a37356624a4027aa89ef0e0fff56', 'image_0222.jpg': '7c94190899d08d22b22554d62996b6f07c6f966aa80a08a017cd77c7a7642305', 'image_0223.jpg': 'aca8f0371e6c80ed4d4084afbbea3b113f11a647d57136f2e4ef200d7861c468', 'image_0229.jpg': '807723f0fbcd0e518473f2b3a6eab39bd12c9c2368b6025fb58d9de8bb135095', 'image_0234.jpg': '4ee930e51d44a0ccdf7845c9dc4d710e30cf2109bbe9985f8ba2e36e09c66a35', 'image_0237.jpg': '856422427e40ec9dbed445f9ce52baa59f0fbfdb330f9e840825682b24c19c5d', 'image_0241.jpg': '5f29c62d3d4eeae6f2710fa3f4e2b342d387ebedb7024d6d3418e800bfdae93f', 'image_0248.jpg': 'ed1f3c6871453baf2b1c973a754915d616f90463916e5034748e862e7d57a15b', 'image_0249.jpg': 'e0d98744c39d22eae82f6ffe4e36debbc18f84919a26a925cd60446ef143d091', 'image_0255.jpg': '6c3c35169b84c046ebc0128c8e356c86f447b2451ac6b80289ab4ed813a6419c', 'image_0260.jpg': '86b17f74e925f3672260f05d026dccb1ba9eb2b3d287a32e99282cb0444082bf'}

In [ ]:
data_dir = next((Path(p).resolve() for p in CFG.DATA_CANDIDATES if (Path(p) / 'train.csv').exists()), None)
assert data_dir is not None, 'Folder data/ tidak ditemukan; atur HOLOGY_DATA_DIR.'
train_df = pd.read_csv(data_dir / 'train.csv')
test_df = pd.read_csv(data_dir / 'test.csv')
sample = pd.read_csv(data_dir / 'sample_submission.csv')
assert len(train_df) == 212 and len(test_df) == 54 and set(sample.image_id) == set(test_df.image_id)
train_df = train_df.merge(pd.read_csv(io.StringIO(FOLD_CSV)), on=['image_id', 'label'], validate='one_to_one')
train_df['split'], test_df['split'], test_df['label'] = 'train', 'test', '?'
df = pd.concat([train_df, test_df], ignore_index=True).merge(pd.read_csv(io.StringIO(TAG_CSV)), on='image_id', how='left', validate='one_to_one')
df['path'] = [data_dir / f'{s}_images' / i for i, s in zip(df.image_id, df.split)]
for p in df.path:
    assert hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_HASHES[p.name], f'Dataset berubah: {p.name}'
n_train = len(train_df)
y = train_df.label.map(CFG.CLASSES.index).to_numpy()
print('Data:', data_dir, '| train', n_train, '| test', len(test_df), '| label', np.bincount(y))

---

# EDA dan Preprocessing <a name="3"></a>

---

Bagian ini merangkum audit CPU (`eda/01` sampai `eda/34`) yang mendasari perubahan v6, lalu menjalankan fungsi preprocessing yang identik dengan yang diuji di CPU. Fungsi disalin otomatis oleh builder dan diverifikasi oleh `eda/test_v6_pipeline.py`.

## Temuan Audit CPU

Tabel pertama merangkum fondasi yang tetap dipakai dari v5, tabel kedua berisi temuan baru setelah v5 dijalankan.

| Fondasi (v5) | Angka | Keputusan |
|---|---|---|
| Format file bocor: Normal = L/q95, abnormal = RGB/q75 | `image.mode` AUC 0.988 | re-encode q75, background = 0, mode tidak dipakai |
| Tiap kasus = CC + MLO | 1-NN tanpa grup 0.780 vs dengan grup 0.555 | CV per komponen grup, fusi per pasangan |
| ColorJitter torchvision | background terangkat 28 sampai 41% | jitter hanya di jaringan ±10% |
| Rotasi naive / reflect / fit | celah dinding dada 28 sampai 88% baris atau jaringan palsu | rotasi cover |

| Temuan setelah v5 | Angka | Keputusan v6 |
|---|---|---|
| Error v5 | Malignant ke Normal 19, ke Benign 19; prediksi Malignant 53 dari 80 di OOF dan 10 dari sekitar 20 di test | kalibrasi bias kelas dengan nested CV |
| DINOv3 | probe 0.47; fusi turun monoton | tidak dipakai |
| Min-max Mammo-FM dikuasai artefak tepi | piksel termaksimal berada di tepi view pada 91 sampai 100% citra; median jaringan citra gelap hanya 0.18 | robust scale: white point = p99.8 jaringan di luar 8 px tepi |
| Sisa jejak format di strip tepi | nilai max (255 vs 254) AUC Normal 0.774 | ter-clip ke 255 oleh robust scale |
| Window p0.5 sampai p99.5 | citra gelap membaik, payudara padat (0087, 0159) jenuh | ditolak, black point tetap 0 |
| Resolusi | kontras titik terang tersisa 85% di 1520x912, 90% di 2048x1228; jumlah titik tidak membedakan kelas (AUC 0.55) | tetap 1520x912 |
| Rotasi cover ±7 pada view Mammo-FM | tepi kulit/puting tersisa 85% rata-rata, minimum 39% | ±3 derajat: tepi tersisa 96% |
| Error sistematis OOF | 63 citra salah di 3/3 run; Malignant ke Benign 22 | head bantu Benign vs Malignant |
| Fitur lesi buatan tangan (area, bentuk, kontras blob) | AUC Benign vs Malignant 0.43 sampai 0.52 | tidak dipakai; batasnya ada pada morfologi yang harus dipelajari model |
| Stacking probabilitas + fitur densitas | nested OOF 0.58 sampai 0.61 vs 0.63 tanpa stacking | tidak dipakai |
| Fusi pasangan dan bias kelas | di bawah noise antar seed (sekitar 0.015) | submission final per citra |

#### Insights

> Perbaikan menyasar penyebab yang terukur. Baseline `robust_rot3` dijalankan pada fold yang sama sehingga selisih skor dapat diatribusikan ke head bantu dan resolusi.

## Fungsi Preprocessing

Base bersama dibuat sekali pada 2048x1536, lalu dipotong ke bbox jaringan dan di-stretch ke 1520x912 sesuai protokol pretraining Mammo-FM. Dua cache disimpan: `raw` (sama dengan v5) dan `robust` (white point robust).

In [ ]:
PREPROCESS_SHA = '3f805d179013a1a31e10a1232cd7633a39b9fe4ee6a37f83af69bf11589501ea'


def breast_mask(a, thr=None):
    b = cv2.GaussianBlur(a, (5, 5), 0)
    otsu = cv2.threshold(b, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0]
    bg = int(np.argmax(np.bincount(b.ravel(), minlength=256)[:max(1, int(otsu))]))
    t = thr if thr is not None else bg + max(15, 0.5 * (otsu - bg))
    m = (b > t).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n < 2:
        return np.ones_like(a, bool)
    k = 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])
    mk = (lab == k).astype(np.uint8)
    cnts, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(mk, cnts, -1, 1, cv2.FILLED)
    return mk.astype(bool)


def read_gray(path, scale=2, harmonize=False):
    with Image.open(path) as image:
        if harmonize:
            buf = io.BytesIO()
            image.convert('L').save(buf, 'JPEG', quality=75)
            buf.seek(0)
            with Image.open(buf) as encoded:
                encoded.draft('L', (encoded.width // scale, encoded.height // scale))
                return np.array(encoded.convert('L'))
        image.draft('L', (image.width // scale, image.height // scale))
        return np.array(image.convert('L'))


def letterbox(array, out=(1024, 768), nearest=False):
    h, w = array.shape
    ratio = min(out[0] / h, out[1] / w)
    hh, ww = (max(1, round(h * ratio)), max(1, round(w * ratio)))
    resized = cv2.resize(array, (ww, hh), interpolation=cv2.INTER_NEAREST if nearest else cv2.INTER_AREA)
    result = np.zeros(out, dtype=array.dtype)
    top = (out[0] - hh) // 2
    result[top:top + hh, :ww] = resized
    return result


def prepare_array(raw, out=(1024, 768), variant='soft', trim=0.0):
    if raw.ndim != 2 or raw.dtype != np.uint8 or min(raw.shape) < 32:
        raise ValueError('Expected a grayscale uint8 image at least 32x32.')
    h, w = raw.shape
    dy, dx = (round(h * trim), round(w * trim))
    a = raw[dy:h - dy, dx:w - dx].copy()
    sh = min(1185, a.shape[0])
    sw = max(32, round(a.shape[1] * sh / a.shape[0]))
    small = cv2.resize(a, (sw, sh), interpolation=cv2.INTER_AREA)
    mask_small = breast_mask(small)
    mask = cv2.resize(mask_small.astype(np.uint8), (a.shape[1], a.shape[0]), interpolation=cv2.INTER_NEAREST)
    flipped = mask[:, :a.shape[1] // 2].sum() < mask[:, a.shape[1] // 2:].sum()
    if flipped:
        a, mask = (a[:, ::-1].copy(), mask[:, ::-1].copy())
    k = max(3, round(15 * a.shape[0] / 1185))
    support = cv2.dilate(mask, np.ones((2 * k + 1, 2 * k + 1), np.uint8))
    distance = cv2.distanceTransform((1 - mask).astype(np.uint8), cv2.DIST_L2, 3)
    alpha = np.clip(1 - distance / k, 0, 1)
    alpha = alpha * alpha * (3 - 2 * alpha)
    if variant == 'raw':
        processed = a
    elif variant == 'hard':
        processed = a * mask
    elif variant in ('soft', 'gamma', 'clahe', 'crop'):
        processed = a.copy()
        if variant == 'gamma':
            processed = np.round((a.astype(np.float32) / 255) ** 0.8 * 255).astype(np.uint8)
        elif variant == 'clahe':
            processed = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(a)
        processed = np.round(processed * alpha).astype(np.uint8)
    else:
        raise ValueError(variant)
    if variant == 'crop':
        yy, xx = np.where(support)
        if not len(yy):
            raise ValueError('Empty support mask.')
        sl = np.s_[yy.min():yy.max() + 1, :xx.max() + 1]
        processed, mask, support = (processed[sl], mask[sl], support[sl])
    image = letterbox(processed, out)
    m = letterbox(mask, out, nearest=True).astype(bool)
    s = letterbox(support, out, nearest=True).astype(bool)
    return (image, m, s, bool(flipped))


def preprocess(path, out=(1024, 768), variant='soft', harmonize=True):
    return prepare_array(read_gray(path, scale=2, harmonize=harmonize), out, variant)


def cover_scale(shape, angle):
    h, w = shape
    rad = np.deg2rad(abs(angle))
    return max((w * np.cos(rad) + h * np.sin(rad)) / w, (w * np.sin(rad) + h * np.cos(rad)) / h)


def rotate_cover(array, angle, nearest=False):
    h, w = array.shape
    m = cv2.getRotationMatrix2D(((w - 1) / 2, (h - 1) / 2), angle, cover_scale(array.shape, angle))
    return cv2.warpAffine(array, m, (w, h), flags=cv2.INTER_NEAREST if nearest else cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


def masked_jitter(array, brightness, contrast):
    tissue = array > 0
    if not tissue.any():
        return array.copy()
    f = array.astype(np.float32)
    mu = f[tissue].mean()
    out = np.clip((f - mu) * contrast + mu * brightness, 1, 255)
    return np.where(tissue, np.round(out), 0).astype(np.uint8)


def augment_v7(array, rng, degrees=7.0, brightness=(0.9, 1.1), contrast=(0.9, 1.1), hflip_p=0.0, vflip_p=0.5):
    a = rotate_cover(array, float(rng.uniform(-degrees, degrees)))
    a = masked_jitter(a, float(rng.uniform(*brightness)), float(rng.uniform(*contrast)))
    if rng.random() < vflip_p:
        a = a[::-1]
    if rng.random() < hflip_p:
        a = a[:, ::-1]
    return np.ascontiguousarray(a)


def fm_view(image, support, out=(1520, 912)):
    ys, xs = np.where(support)
    if not len(ys):
        raise ValueError('Empty support mask.')
    crop = image[ys.min():ys.max() + 1, :xs.max() + 1]
    return cv2.resize(np.ascontiguousarray(crop), (out[1], out[0]), interpolation=cv2.INTER_AREA)


def robust_scale(array, pct=99.8, border=8):
    inner = np.zeros(array.shape, bool)
    inner[border:-border, border:-border] = True
    tissue = array[(array > 0) & inner]
    hi = np.percentile(tissue, pct) if tissue.size else 255.0
    return np.where(array > 0, np.clip(array.astype(np.float32) / max(hi, 1.0) * 255, 1, 255), 0).astype(np.uint8)

In [ ]:
def build_view(path):
    image, mask, support, flipped = preprocess(path, (CFG.BASE_H, CFG.BASE_W), 'soft', True)
    return fm_view(image, support, (CFG.FM_H, CFG.FM_W)), flipped


t0 = time.time()
with ThreadPoolExecutor(CFG.PREP_WORKERS) as pool:
    built = list(pool.map(build_view, df.path))
IMAGES = {'raw': np.stack([b[0] for b in built])}
df['flipped'] = [b[1] for b in built]
del built
with ThreadPoolExecutor(CFG.PREP_WORKERS) as pool:
    IMAGES['robust'] = np.stack(list(pool.map(robust_scale, IMAGES['raw'])))


def build_hi_view(path):
    image, mask, support, _ = preprocess(path, CFG.HI_BASE, 'soft', True)
    return robust_scale(fm_view(image, support, (CFG.HI_H, CFG.HI_W)))


if any(e['view'] == 'robust_hi' for e in CFG.EXPERIMENTS):
    with ThreadPoolExecutor(CFG.PREP_WORKERS) as pool:
        IMAGES['robust_hi'] = np.stack(list(pool.map(build_hi_view, df.path)))
print({k: v.shape for k, v in IMAGES.items()}, f'{time.time() - t0:.0f}s')

## Pemeriksaan Visual dan Statistik per Kelas

Setiap kelas ditampilkan dalam view v5 (min-max), view v6 (robust), dan dua augmentasi v6. Statistik jaringan per kelas harus serupa antar split; background harus tetap 0.

In [ ]:
def minmax_u8(a):
    f = a.astype(np.float32)
    return ((f - f.min()) / max(f.max() - f.min(), 1) * 255).astype(np.uint8)


rng = np.random.default_rng(CFG.SEED)
dark = np.argsort([np.median(minmax_u8(a)[a > 0]) for a in IMAGES['raw'][:n_train]])[:2]
picks = list(dark) + [i for c in range(3) for i in np.flatnonzero(y == c)[:2]]
fig, axes = plt.subplots(len(picks), 4, figsize=(11, 4.2 * len(picks)))
for r, i in enumerate(picks):
    shown = [minmax_u8(IMAGES['raw'][i]), IMAGES['robust'][i],
             augment_v7(IMAGES['robust'][i], rng, 3.0, CFG.JITTER, CFG.JITTER, 0.0, CFG.VFLIP_P),
             augment_v7(IMAGES['robust'][i], rng, 3.0, CFG.JITTER, CFG.JITTER, 0.0, CFG.VFLIP_P)]
    for c, a in enumerate(shown):
        axes[r, c].imshow(a, cmap='gray', vmin=0, vmax=255)
        axes[r, c].set_title(f"{df.image_id[i][6:10]} {CFG.CLASSES[y[i]]} / {['v5 min-max', 'v6 robust', 'v6 aug', 'v6 aug'][c]}", fontsize=8)
        axes[r, c].axis('off')
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'views_per_class.png', dpi=50)
plt.show()
view_stats = pd.DataFrame({'split': df.split, 'label': df.label,
                           'p50_v5': [np.median(minmax_u8(a)[a > 0]) for a in IMAGES['raw']],
                           'p50_v6': [np.median(a[a > 0]) for a in IMAGES['robust']],
                           'saturated_v6': [(a[a > 0] >= 254).mean() for a in IMAGES['robust']],
                           'background_nonzero': [(a[:, -16:] > 0).mean() for a in IMAGES['robust']]})
display(view_stats.groupby(['split', 'label']).median().round(3))

#### Insights

> Citra gelap pada kolom v5 terlihat pudar karena satu strip terang di tepi menentukan skala; kolom v6 mengembalikan kontras jaringan tanpa menjenuhkan payudara padat. Saturasi v6 sekitar 0.4% jaringan, sesuai white point p99.8.

---

# Validasi dan Pasangan View <a name="4"></a>

---

Fold diambil dari manifest grup hasil audit yang sama dengan v5. Pasangan CC-MLO untuk fusi prediksi dicari dari fitur Mammo-FM beku dengan batasan view berbeda, laterality sama, dan split sama.

## Audit Fold

Cell berikut memastikan tidak ada grup yang terbelah antar fold dan setiap fold memuat ketiga kelas.

In [ ]:
groups = train_df.group_union.to_numpy()
fold_id = train_df.fold_group_union.to_numpy()
FOLDS = [(np.flatnonzero(fold_id != f), np.flatnonzero(fold_id == f)) for f in range(CFG.N_FOLDS)]
for a, b in FOLDS:
    assert not set(groups[a]) & set(groups[b]) and set(y[b]) == {0, 1, 2}
display(pd.crosstab(train_df.fold_group_union, train_df.label))

## Encoder Mammo-FM dan Pasangan CC-MLO

Checkpoint diunduh sekali dan hanya bobot image encoder yang diekstrak. Fitur beku dipakai untuk Hungarian matching di setiap sel (split, laterality).

In [ ]:
enc_path = CFG.WEIGHTS_DIR / 'mammo_fm_encoder.pt'
if not enc_path.exists():
    ckpt_path = hf_hub_download(CFG.FM_REPO, CFG.FM_FILE, local_dir=CFG.WEIGHTS_DIR / 'mammo_fm')
    ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    torch.save({k[len('image_encoder.'):]: v for k, v in ck['model'].items() if k.startswith('image_encoder.')}, enc_path)
    del ck
    gc.collect()


def build_fm_encoder():
    net = EfficientNet.from_name('efficientnet-b5')
    missing, unexpected = net.load_state_dict(torch.load(enc_path, map_location='cpu'), strict=False)
    assert all(k.startswith('_fc') for k in missing) and not unexpected, (missing, unexpected)
    return net


def fm_tensor(batch):
    x = torch.as_tensor(np.ascontiguousarray(batch), device='cuda', dtype=torch.float32)[:, None]
    lo = x.amin(dim=(-2, -1), keepdim=True)
    hi = x.amax(dim=(-2, -1), keepdim=True)
    x = ((x - lo) / (hi - lo).clamp(min=1e-6) - CFG.FM_MEAN) / CFG.FM_STD
    return x.expand(-1, 3, -1, -1).contiguous(memory_format=torch.channels_last)


def fm_pool(net, x):
    return net._avg_pooling(net.extract_features(x)).flatten(1)


@torch.inference_mode()
def fm_features(images, bs=16):
    net = build_fm_encoder().cuda().eval().to(memory_format=torch.channels_last)
    out = []
    for k in range(0, len(images), bs):
        with torch.autocast('cuda', dtype=CFG.AMP_DTYPE):
            out.append(fm_pool(net, fm_tensor(images[k:k + bs])).float().cpu().numpy())
    del net
    torch.cuda.empty_cache()
    return np.concatenate(out)


z = StandardScaler().fit_transform(fm_features(IMAGES['robust']))
z /= np.linalg.norm(z, axis=1, keepdims=True)
sim = z @ z.T
df['pair'] = np.arange(len(df))
for _, cell in df.groupby(['split', 'lat2']):
    cc = cell.index[cell.view2 == 'CC'].to_numpy()
    ml = cell.index[cell.view2 == 'MLO'].to_numpy()
    if len(cc) and len(ml):
        r, c = linear_sum_assignment(-sim[np.ix_(cc, ml)])
        df.loc[ml[c], 'pair'] = cc[r]
paired = df[df.duplicated('pair', keep=False)]
same = paired[paired.split == 'train'].groupby('pair').label.nunique().eq(1).mean()
print('pairs:', paired.pair.nunique(), '| train same-label rate:', round(same, 3), '| test pairs:', paired[paired.split == 'test'].pair.nunique())

#### Insights

> Same-label rate pasangan train di atas 0.9 menandakan pencocokan layak untuk fusi. Pasangan hanya dipakai untuk merata-ratakan prediksi, tidak pernah untuk memindahkan label.

---

# Fine-tuning Mammo-FM <a name="5"></a>

---

Setiap eksperimen melatih 5 model fold dengan resep v5: encoder penuh dengan BatchNorm beku, head dropout + linear, CE berbobot kelas dengan label smoothing, AdamW, warmup lalu cosine, dan model epoch terakhir. Yang berbeda antar eksperimen hanya cache intensitas, sudut rotasi, dan seed.

## Dataset dan Model

Augmentasi dijalankan di worker DataLoader dengan RNG per worker. TTA memakai citra asli dan vertical flip; horizontal flip tidak dipakai karena dinding dada selalu di kiri.

In [ ]:
class Mammograms(Dataset):
    def __init__(self, images, labels, rot):
        self.images, self.labels, self.rot, self.rng = images, labels, rot, None

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        if self.rng is None:
            self.rng = np.random.default_rng(torch.initial_seed() % 2 ** 32)
        a = augment_v7(self.images[i], self.rng, self.rot, CFG.JITTER, CFG.JITTER, 0.0, CFG.VFLIP_P)
        return torch.from_numpy(a), int(self.labels[i])


class FMClassifier(nn.Module):
    def __init__(self, bm_aux=False):
        super().__init__()
        self.encoder = build_fm_encoder()
        self.head = nn.Sequential(nn.Dropout(CFG.DROP), nn.Linear(2048, 3))
        self.head_bm = nn.Sequential(nn.Dropout(CFG.DROP), nn.Linear(2048, 1)) if bm_aux else None

    def train(self, mode=True):
        super().train(mode)
        for m in self.encoder.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def forward(self, x):
        f = fm_pool(self.encoder, x)
        return self.head(f), (self.head_bm(f).squeeze(1) if self.head_bm is not None else None)

    def probs(self, x):
        logits, bm = self(x)
        p = logits.float().softmax(1)
        if bm is None:
            return p
        s = bm.float().sigmoid()
        split = torch.stack([p[:, 0], (1 - p[:, 0]) * (1 - s), (1 - p[:, 0]) * s], 1)
        return (p + split) / 2


@torch.inference_mode()
def predict(model, images, bs=4):
    model.eval()
    out = []
    for k in range(0, len(images), bs):
        x = fm_tensor(images[k:k + bs])
        with torch.autocast('cuda', dtype=CFG.AMP_DTYPE):
            p = model.probs(x) + model.probs(x.flip(-2))
        out.append((p / 2).cpu().numpy())
    return np.concatenate(out)

## Training Loop

Checkpoint per eksperimen dan fold disimpan, sehingga run yang terputus dapat dilanjutkan dan mode inference dapat memakai ulang bobot. Probabilitas OOF dan test per citra disimpan ke CSV, tidak seperti v5.

In [ ]:
def fit_fold(exp, fold, a, b):
    images = IMAGES[exp['view']]
    accum = CFG.EFFECTIVE_BATCH // exp['batch']
    seed_everything(CFG.SEED + 100 * exp['seed'] + fold)
    model = FMClassifier(exp['bm_aux']).cuda().to(memory_format=torch.channels_last)
    ckpt = CFG.OUTPUT_DIR / f"{exp['name']}_fold{fold}.pt"
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location='cuda', weights_only=True))
        print('reuse', ckpt.name)
    else:
        assert CFG.RUN_MODE == 'train', f'Checkpoint belum ada: {ckpt}'
        loader = DataLoader(Mammograms(images[a], y[a], exp['rot']), batch_size=exp['batch'], shuffle=True, drop_last=True,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True)
        opt = torch.optim.AdamW([{'params': model.encoder.parameters(), 'lr': CFG.LR_ENCODER},
                                 {'params': [p for n, p in model.named_parameters() if n.startswith('head')], 'lr': CFG.LR_HEAD}],
                                weight_decay=CFG.WEIGHT_DECAY)
        steps = math.ceil(len(loader) / accum) * CFG.EPOCHS
        warm = max(1, int(steps * CFG.WARMUP))
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * min(1, (s - warm) / max(1, steps - warm)))))
        weights = torch.tensor(len(a) / (3 * np.bincount(y[a], minlength=3)), device='cuda', dtype=torch.float32)
        criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=CFG.SMOOTHING)
        bm_criterion = nn.BCEWithLogitsLoss()
        history = []
        for epoch in range(CFG.EPOCHS):
            model.train()
            opt.zero_grad(set_to_none=True)
            t0, total = time.time(), 0.0
            for step, (x, t) in enumerate(loader):
                with torch.autocast('cuda', dtype=CFG.AMP_DTYPE):
                    logits, bm = model(fm_tensor(x.numpy()))
                    t = t.cuda()
                    loss = criterion(logits.float(), t)
                    abnormal = t > 0
                    if bm is not None and abnormal.any():
                        loss = loss + CFG.BM_WEIGHT * bm_criterion(bm[abnormal].float(), (t[abnormal] == 2).float())
                assert torch.isfinite(loss)
                (loss / accum).backward()
                total += loss.item()
                if (step + 1) % accum == 0 or step + 1 == len(loader):
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                    opt.zero_grad(set_to_none=True)
                    sched.step()
            val = predict(model, images[b])
            history.append({'epoch': epoch + 1, 'loss': total / len(loader), 'val_f1': f1_score(y[b], val.argmax(1), average='macro'),
                            'seconds': time.time() - t0})
            print(exp['name'], fold, {k: round(v, 4) for k, v in history[-1].items()})
        pd.DataFrame(history).to_csv(CFG.OUTPUT_DIR / f"history_{exp['name']}_{fold}.csv", index=False)
        torch.save(model.state_dict(), ckpt)
    val, test = predict(model, images[b]), predict(model, images[n_train:])
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return val, test


def save_probs(name, oof, test):
    o = train_df[['image_id', 'label']].copy()
    t = test_df[['image_id']].copy()
    for i, c in enumerate(CFG.CLASSES):
        o[f'p_{c}'], t[f'p_{c}'] = oof[:, i], test[:, i]
    o.to_csv(CFG.OUTPUT_DIR / f'oof_{name}.csv', index=False)
    t.to_csv(CFG.OUTPUT_DIR / f'test_{name}.csv', index=False)


OOF, TEST = {}, {}
for exp in CFG.EXPERIMENTS:
    oof, test = np.zeros((n_train, 3)), np.zeros((len(df) - n_train, 3))
    for fold, (a, b) in enumerate(FOLDS):
        oof[b], p = fit_fold(exp, fold, a, b)
        test += p / CFG.N_FOLDS
    OOF[exp['name']], TEST[exp['name']] = oof, test
    save_probs(exp['name'], oof, test)
    print(exp['name'], 'OOF macro F1', round(f1_score(y, oof.argmax(1), average='macro'), 4))

#### Insights

> Selisih `robust_rot3` dan `robust_rot3_s1` menunjukkan noise antar seed. `bm_aux`, `hires`, dan `hires_bm_aux` hanya dianggap lebih baik bila selisihnya terhadap baseline melebihi noise tersebut dan AUC Benign vs Malignant ikut naik.

---

# Analisis Error <a name="6"></a>

---

Analisis error menghubungkan kesalahan OOF dengan properti citra. Korelasi Spearman antara indikator error dan fitur citra menunjukkan kondisi apa yang membuat model gagal, sehingga perbaikan berikutnya dapat diarahkan ke penyebab, bukan ditebak.

## Correlation Matrix Error vs Properti Citra

Fitur dihitung dari view mentah (sebelum robust scale): kecerahan jaringan, headroom artefak tepi, fill ratio, laterality, view, flip, dan ketidaksepakatan prediksi dalam pasangan CC-MLO.

In [ ]:
def fuse_pairs(probs, frame):
    return pd.DataFrame(probs).groupby(frame.pair.to_numpy()).transform('mean').to_numpy()


def macro(probs):
    return f1_score(y, probs.argmax(1), average='macro')


main_name = max(OOF, key=lambda k: macro(OOF[k]))
P = OOF[main_name]
pred = P.argmax(1)
raw = IMAGES['raw'][:n_train]
tr = df.iloc[:n_train]
partner = fuse_pairs(np.eye(3)[pred], tr)
feat = pd.DataFrame({
    'tissue_p50_v5': [np.median(minmax_u8(a)[a > 0]) for a in raw],
    'edge_headroom': [a.max() - np.percentile(a[a > 0], 99.9) for a in raw],
    'fill_ratio': (raw > 0).mean((1, 2)),
    'is_cc': (tr.view2 == 'CC').astype(int).to_numpy(),
    'is_right': (tr.lat2 == 'R').astype(int).to_numpy(),
    'flipped': tr.flipped.astype(int).to_numpy(),
    'pair_disagree': 1 - partner.max(1),
    'p_malignant': P[:, 2],
})
target = pd.DataFrame({'wrong': (pred != y).astype(int), 'p_true': P[np.arange(n_train), y],
                       'malignant_missed': ((y == 2) & (pred != 2)).astype(int), 'malignant_to_normal': ((y == 2) & (pred == 0)).astype(int)})
corr = pd.concat([feat, target], axis=1).corr(method='spearman').loc[feat.columns, target.columns]
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=30)
ax.set_yticks(range(len(corr.index)), corr.index)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im)
ax.set_title(f'Spearman: error vs properti citra ({main_name})')
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'error_correlation.png', dpi=80)
plt.show()
print('Error rate per class:', {c: round(float((pred[y == k] != k).mean()), 3) for k, c in enumerate(CFG.CLASSES)})

## Kesalahan dengan Confidence Tertinggi

Dua belas kesalahan dengan probabilitas prediksi tertinggi ditampilkan beserta label asli, sebagai pemeriksaan visual apakah errornya masuk akal secara klinis (misalnya BI-RADS 3 vs 4) atau karena artefak.

In [ ]:
wrong = np.flatnonzero(pred != y)
worst = wrong[np.argsort(-P[wrong].max(1))][:12]
fig, axes = plt.subplots(2, 6, figsize=(18, 10))
for ax, i in zip(axes.flat, worst):
    ax.imshow(IMAGES['robust'][i], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f"{tr.image_id.iloc[i][6:10]} true {CFG.CLASSES[y[i]]}\npred {CFG.CLASSES[pred[i]]} {P[i].max():.2f}", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'confident_errors.png', dpi=60)
plt.show()

#### Insights

> Korelasi positif `malignant_to_normal` dengan `edge_headroom` atau korelasi negatif dengan `tissue_p50_v5` berarti error masih dipicu citra gelap; korelasi dengan `pair_disagree` berarti fusi pasangan akan membantu. Nilai absolut di bawah sekitar 0.14 tidak signifikan untuk 212 citra.

---

# Kalibrasi, Ensemble, dan Submission <a name="7"></a>

---

v5 memprediksi Malignant terlalu sedikit di OOF (53 dari 80) maupun di test (10 dari sekitar 20). Bias log-probabilitas per kelas dapat menggeser keputusan tanpa melatih ulang, tetapi bias yang dipilih dari OOF yang sama akan optimistis. Karena itu skornya diukur dengan nested CV: bias dipilih dari empat fold dan diterapkan ke fold kelima.

## Kandidat dan Bias Kelas Nested

Kandidat terdiri dari setiap eksperimen, ensemble seed robust, dan ensemble semua eksperimen, masing-masing per citra dan setelah fusi pasangan. Bias Normal ditetapkan 0; bias Benign dan Malignant dicari pada grid.

In [ ]:
def best_bias(probs, idx):
    logp = np.log(np.clip(probs[idx], 1e-8, 1))
    best, score = (0.0, 0.0), -1
    for bb in CFG.BIAS_GRID:
        for bm in CFG.BIAS_GRID:
            s = f1_score(y[idx], (logp + [0, bb, bm]).argmax(1), average='macro')
            if s > score + 1e-9:
                best, score = (bb, bm), s
    return np.array([0.0, *best])


def nested_bias_oof(probs):
    out = np.zeros(n_train, int)
    for a, b in FOLDS:
        bias = best_bias(probs, a)
        out[b] = (np.log(np.clip(probs[b], 1e-8, 1)) + bias).argmax(1)
    return out


CANDIDATES = dict(OOF)
CANDIDATES_TEST = dict(TEST)
for ens_name, members in {'ens_base': ['robust_rot3', 'robust_rot3_s1'], 'ens_new': ['bm_aux', 'hires', 'hires_bm_aux']}.items():
    members = [k for k in members if k in OOF]
    if len(members) > 1:
        CANDIDATES[ens_name] = np.mean([OOF[k] for k in members], 0)
        CANDIDATES_TEST[ens_name] = np.mean([TEST[k] for k in members], 0)
CANDIDATES['ens_all'] = np.mean(list(OOF.values()), 0)
CANDIDATES_TEST['ens_all'] = np.mean(list(TEST.values()), 0)
te_frame = df.iloc[n_train:]
rows = []
for name, probs in CANDIDATES.items():
    for pair in [False, True]:
        p = fuse_pairs(probs, tr) if pair else probs
        bm = y > 0
        rows.append({'model': name, 'pair_fusion': pair, 'macro_f1': macro(p),
                     'macro_f1_bias_nested': f1_score(y, nested_bias_oof(p), average='macro'),
                     'auc_bm': roc_auc_score(y[bm] == 2, p[bm, 2] / np.maximum(p[bm, 1:].sum(1), 1e-8)),
                     'pred_malignant': int((p.argmax(1) == 2).sum())})
table = pd.DataFrame(rows).round(4).sort_values('macro_f1', ascending=False)
table.to_csv(CFG.OUTPUT_DIR / 'model_comparison.csv', index=False)
display(table)

#### Insights

> `macro_f1_bias_nested` adalah estimasi jujur manfaat kalibrasi. Jika nilainya tidak melebihi `macro_f1` tanpa bias, kekurangan prediksi Malignant bukan sekadar masalah threshold dan bias tidak dipakai.

## Pemilihan dan Submission

Submission final dipilih dari kandidat per citra (tanpa fusi pasangan dan tanpa bias kelas). Pasangan test hanya dapat ditebak dari kemiripan fitur tanpa verifikasi, sehingga satu pasangan yang salah mencampur prediksi dua pasien berbeda; di OOF, manfaat fusi (sekitar 0.008) dan bias nested masih di bawah noise antar seed (sekitar 0.015). Varian fusi dan bias tetap disimpan sebagai CSV terpisah untuk perbandingan.

In [ ]:
def save_submission(probs, filename, bias=None):
    logp = np.log(np.clip(probs, 1e-8, 1)) + (0 if bias is None else bias)
    mapping = dict(zip(te_frame.image_id, np.asarray(CFG.CLASSES)[logp.argmax(1)]))
    sub = sample[['image_id']].copy()
    sub['label'] = sub.image_id.map(mapping)
    assert sub.label.isin(CFG.CLASSES).all() and len(sub) == 54
    sub.to_csv(CFG.OUTPUT_DIR / filename, index=False)
    return sub


for name, probs in CANDIDATES_TEST.items():
    save_submission(probs, f'submission_{name}.csv')
    save_submission(fuse_pairs(probs, te_frame), f'submission_{name}_pair.csv')
best = table[~table.pair_fusion].sort_values('macro_f1', ascending=False).iloc[0]
oof_best, test_best, bias = CANDIDATES[best.model], CANDIDATES_TEST[best.model], None
sub = save_submission(test_best, 'submission.csv')
print('selected:', best.model, '| pair fusion: False | bias: None | OOF macro F1:', best.macro_f1)
print('test label counts:', sub.label.value_counts().to_dict())
final_oof = (np.log(np.clip(oof_best, 1e-8, 1)) + (0 if bias is None else bias)).argmax(1)
ConfusionMatrixDisplay(confusion_matrix(y, final_oof), display_labels=CFG.CLASSES).plot(colorbar=False)
plt.title(f'OOF {best.model}')
plt.savefig(CFG.OUTPUT_DIR / 'confusion_best.png', dpi=80)
plt.show()

#### Insights

> Confusion matrix di atas adalah OOF kandidat terpilih tanpa fusi dan tanpa bias. Public leaderboard dan skor 0.721 v5 dihitung pada 54 citra, sehingga satu citra menggeser skor sekitar 0.02 sampai 0.03; pemilihan mengikuti OOF, bukan leaderboard.